# 13 Hands-on Vector Databases: FAISS, Elasticsearch, Milvus & pgvector

**Real-World Scenario**: Multi-Database Enterprise Storage Evaluation (FAISS, `pgvector`, Elasticsearch).

This notebook demonstrates **Product Quantization (PQ-64) Memory Compression Math** ($96	ext{x}$ RAM reduction) and persists indexes in a hidden `.vectordb/` directory.

## Step 1: Environment Setup

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from repository root .env
load_dotenv()

# Create hidden vector database directory for local index persistence
os.makedirs(".vectordb", exist_ok=True)

print("=== Repository Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))
print("Hidden DB Directory Path:", os.path.abspath(".vectordb"))


## Step 2: Product Quantization (PQ-64) Memory Savings Math

In [ ]:
def calculate_pq_savings(num_vectors=1_000_000, dim=1536, sub_vecs=64):
    raw_fp32_gb = (num_vectors * dim * 4) / (1024**3)
    pq_uint8_gb = (num_vectors * sub_vecs * 1) / (1024**3)
    
    print(f"=== Product Quantization (PQ-64) Memory Math ({num_vectors:,} Vectors) ===")
    print(f"Raw FP32 Memory Footprint: {raw_fp32_gb:.2f} GB")
    print(f"PQ-64 Memory Footprint:    {pq_uint8_gb:.2f} GB")
    print(f"Memory Reduction Factor:   {raw_fp32_gb / pq_uint8_gb:.1f}x RAM Savings!")

calculate_pq_savings()


## Step 3: FAISS Hidden Directory Index Persistence

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

docs = ["Microservice A communicates with Database B via gRPC over TLS 1.3."]

if os.getenv("OPENAI_API_KEY"):
    vectorstore = FAISS.from_texts(docs, OpenAIEmbeddings())
    save_path = os.path.join(".vectordb", "13_multidb_faiss_index")
    vectorstore.save_local(save_path)
    
    results = vectorstore.similarity_search("gRPC protocol", k=1)
    print("=== FAISS Index Search Result ===")
    print(results[0].page_content)
    print(f"Saved index into hidden folder: {save_path}")
